# Data preprocessing

Research code from my MSc thesis, University of Copenhagen, 2024.

Original research cells are preserved; saved outputs are cleared. See the repository README and docs/source_notes.md for inputs and execution context.


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys, os
from time import time
from pathlib import Path
from itertools import product
from tqdm import tqdm

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import time

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dot, Reshape, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.callbacks import EarlyStopping
import seaborn as sns
from sklearn.preprocessing import quantile_transform
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

from scipy.stats import spearmanr

from sklearn.preprocessing import quantile_transform


from numpy.linalg import inv

from joblib import Parallel, delayed
from sklearn.preprocessing import quantile_transform, StandardScaler, MinMaxScaler


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install pandas pyarrow

import pandas as pd

# Define the file path
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'

# Load the Parquet file into a DataFrame
df = pd.read_parquet(file_path, engine='pyarrow')

# Check the loaded data
print(df.head())


In [ ]:
import pandas as pd

# Assuming df is your DataFrame

# Rename the 'month' column to 'date'
df = df.rename(columns={'month': 'date'})

# Ensure 'df' has a MultiIndex ['date', 'permno']
df = df.set_index(['date', 'permno'])

# Sort the MultiIndex
df = df.sort_index()


# Step 1: Calculate the number of stocks for each month
df['num_stocks'] = df.groupby('date')['ret_excess'].transform('count')

# Step 2: Calculate the equal weight for each stock
df['equal_weight'] = 1 / df['num_stocks']

# Step 3: Calculate the equal-weighted portfolio return for each stock
df['equal_weighted_return'] = df['ret_excess'] * df['equal_weight']

# Step 4: Aggregate to get the equal-weighted portfolio return for each month
equal_weighted_portfolio = df.groupby('date')['equal_weighted_return'].sum()

# Join the equal_weighted_portfolio back to the original DataFrame
df = df.join(equal_weighted_portfolio.rename('equal_weighted_portfolio'), on='date')
# Rename the 'equal_weighted_portfolio' column
df.rename(columns={'equal_weighted_portfolio': 'characteristic_equal_weighted_portfolio'}, inplace=True)


# Display the updated DataFrame to check the results
print(df.head())


In [ ]:
pip install parallelbar


In [ ]:


# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade", "equal_weighted_portfolio"]
# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]


# Filter for dates starting from '1993'
df = df.loc[pd.IndexSlice['1960':, :], :]

# Create forward returns
df['fwd_ret_excess'] = df['ret_excess'].unstack('permno').shift(-1).stack(dropna=False)




# # Normalize the characteristics columns
# df.loc[:, characteristics] = (df.loc[:, characteristics]
#                                           .groupby(level='date')
#                                           .apply(lambda x: pd.DataFrame(quantile_transform(x,
#                                                                                            copy=True,
#                                                                                            n_quantiles=x.shape[0]),
#                                                                         columns=characteristics,
#                                                                         index=x.index.get_level_values('permno')))
#                                          .mul(2).sub(1))





import pandas as pd
import numpy as np
from sklearn.preprocessing import quantile_transform, StandardScaler
from tqdm import tqdm
from joblib import Parallel, delayed

# Define the function to apply quantile transformation for a single date
def process_date(date, df, characteristics):
    group = df.loc[date, characteristics]
    transformed = quantile_transform(group, copy=True, n_quantiles=group.shape[0])
    transformed_df = pd.DataFrame(transformed, columns=characteristics, index=group.index)
    transformed_df = transformed_df.mul(2).sub(1)
    transformed_df['date'] = date  # Add the date back to the DataFrame
    return transformed_df

# Initialize the progress bar
dates = df.index.get_level_values('date').unique()
pbar = tqdm(dates, desc="Processing Dates")

# Apply the transformation with parallel processing
normalized_data = Parallel(n_jobs=-1)(delayed(process_date)(date, df, characteristics) for date in pbar)

# Concatenate all the normalized data back into a single DataFrame
normalized_df = pd.concat(normalized_data)

# Reset and set the index to match the original DataFrame's multi-index
normalized_df = normalized_df.reset_index().set_index(['date', 'permno'])

# Ensure the index order matches the original DataFrame
normalized_df = normalized_df.reindex(df.index)

# Assign the normalized values back to the original DataFrame
df.loc[:, characteristics] = normalized_df[characteristics].values

# Close the progress bar
pbar.close()

# Scale the variables
scaler = StandardScaler()
df[characteristics] = scaler.fit_transform(df[characteristics])

# Clip the returns columns
df.loc[:, ['ret_excess', 'fwd_ret_excess']] = df.loc[:, ['ret_excess', 'fwd_ret_excess']].clip(lower=-1, upper=1.0)


# Print the first few rows of the scaled DataFrame to verify
print(df.head())


In [ ]:
print(df.index)

In [ ]:
print(df)

In [ ]:

# Add 'ret_excess' to the list of columns
columns_to_select = characteristics

# Select these columns from the DataFrame
characteristics_data = df[columns_to_select]

# Define a function to fill missing values with the median for each group
def fill_missing_with_median(group):
    return group.fillna(group.median())

# Apply the function to each group (each date) and preserve the index
df_filled = characteristics_data.groupby(level='date').apply(fill_missing_with_median)

# Remove the additional 'date' level in the index introduced by groupby
df_filled.index = df_filled.index.droplevel(0)

# Add the filled columns back to the original DataFrame
df.update(df_filled)

print(df)

# Display the updated DataFrame to check the results
print(df.head())

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# File path to the Parquet file
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/x_t_preprocessed2.parquet'


# Now save to Parquet again
xt_scaled.to_parquet(file_path)


**porper way**

In [ ]:
import pandas as pd
import numpy as np

columns_to_select = characteristics + ["ret_excess"]
characteristics_data = df[columns_to_select]

# Function to calculate x_t for a given month using Ridge Regression
def calculate_xt(df, date, alpha=0):
    # Filter DataFrame for the given date
    df_month = df.loc[date]

    # Separate returns and characteristics
    r_t = df_month['ret_excess'].values
    characteristic_columns = df_month.drop(columns=['ret_excess']).columns
    Z_t_minus_1 = df_month.drop(columns=['ret_excess']).values

    # Calculate (Z_{t-1}'Z_{t-1})^{-1}Z_{t-1}r_t using Ridge Regression
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])

    Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)

    # Calculate x_t
    x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t

    return x_t, characteristic_columns

# Calculate x_t for all unique dates in the dataset
def calculate_xt_all_dates(df, alpha=1e-6):
    xt_results = {}
    characteristic_columns = None

    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t, characteristic_columns = calculate_xt(df, date, alpha)
        xt_results[date] = x_t

    # Convert dictionary to DataFrame
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')

    # Assign original characteristic names as column names
    if characteristic_columns is not None:
        xt_df.columns = characteristic_columns

    # Preserve the index name
    xt_df.index.name = 'date'

    return xt_df

# Example usage
xt_all_dates = calculate_xt_all_dates(characteristics_data)

# # Scaling the returns
# scaler = StandardScaler()
# xt_scaled = pd.DataFrame(scaler.fit_transform(xt_all_dates), index=xt_all_dates.index, columns=xt_all_dates.columns)


# Print the results for the first few dates to confirm
print(xt_all_dates.head())


**not so proper way**

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
# Create an empty DataFrame to store the returns of the characteristics-managed portfolios
dates = characteristics_data.index.get_level_values(0).unique()
char_managed_portfolios = pd.DataFrame(index=dates, columns=characteristics)

for characteristic in tqdm(characteristics):
    # Rank the stocks for each date based on the characteristic
    characteristics_data[f'rank_{characteristic}'] = characteristics_data.groupby(level='date')[characteristic].rank(ascending=False)

    # Define the top and bottom deciles
    def top_decile(x):
        return x <= x.quantile(0.1)

    def bottom_decile(x):
        return x >= x.quantile(0.9)

    characteristics_data[f'top_decile_{characteristic}'] = characteristics_data.groupby(level='date')[f'rank_{characteristic}'].transform(top_decile)
    characteristics_data[f'bottom_decile_{characteristic}'] = characteristics_data.groupby(level='date')[f'rank_{characteristic}'].transform(bottom_decile)

    # Calculate the returns for the long-short portfolio
    top_decile_mask = characteristics_data[f'top_decile_{characteristic}']
    bottom_decile_mask = characteristics_data[f'bottom_decile_{characteristic}']

    long_returns = characteristics_data.loc[top_decile_mask, 'ret_excess']
    short_returns = characteristics_data.loc[bottom_decile_mask, 'ret_excess']

    long_short_returns = long_returns.groupby(level='date').mean() - short_returns.groupby(level='date').mean()

    # Store the results in the DataFrame
    char_managed_portfolios[characteristic] = long_short_returns

# Replace NaN values with 0 if there are any
char_managed_portfolios.fillna(0, inplace=True)

print(char_managed_portfolios)


In [ ]:
import pandas as pd

# Assuming df is your DataFrame already loaded

# Define the columns to select
columns_to_select = characteristics + ["ret_excess", "fwd_ret_excess"]
characteristics_data = df[columns_to_select]

# Sort the MultiIndex
df_filled = characteristics_data.sort_index()

# Split the DataFrame into characteristics and returns
characteristics_df = df_filled[characteristics]
returns_df = df_filled[['ret_excess']]

# Step 1: Unstack the 'permno' level for returns DataFrame
returns_unstacked = returns_df.unstack(level='permno')

# Step 2: Fill NaN values with -2 in the returns DataFrame
returns_unstacked_filled = returns_unstacked.fillna(-2)

# Step 3: Stack the returns DataFrame back
returns_stacked = returns_unstacked_filled.stack(level='permno')

# Rename the column back to 'ret_excess'
returns_stacked.name = 'ret_excess'

# Convert the stacked returns DataFrame to a DataFrame for joining
returns_stacked_df = returns_stacked.reset_index()

# Convert the original characteristics DataFrame to a DataFrame for joining
characteristics_df = characteristics_df.reset_index()

# Include 'fwd_ret_excess' in the characteristics_df
# Since 'fwd_ret_excess' has only NaN values, we can directly add it from the original df_filled
characteristics_df['fwd_ret_excess'] = df_filled['fwd_ret_excess'].values

# Step 4: Perform a left join to merge the characteristics DataFrame back with the modified returns DataFrame
final_df = pd.merge(returns_stacked_df, characteristics_df, on=['date', 'permno'], how='left')

# Set the index back to MultiIndex for consistency
final_df.set_index(['date', 'permno'], inplace=True)

# Display the final DataFrame with NaNs filled in the returns column
print("Final DataFrame with NaNs filled in 'ret_excess':")
print(final_df)

# Count the number of unique permnos for each date in the final DataFrame
permno_counts = final_df.reset_index().groupby('date')['permno'].nunique()

# Check if the number of permnos is the same for each date
is_same_permno_count = permno_counts.nunique() == 1

# Display the result
print("Number of unique permnos for each date:")
print(permno_counts)
print("\nIs the number of permnos the same for each date?:", is_same_permno_count)


In [ ]:
import pandas as pd

# Assuming df is your DataFrame

# Method 1: Using isna() or isnull()
nan_mask = final_df['fwd_ret_excess'].isna()

# Method 2: Counting the number of NaN values
nan_count = final_df['fwd_ret_excess'].isna().sum()
print(f"Number of NaN values in 'fwd_ret_excess': {nan_count}")

# Method 3: Checking if there are any NaN values
has_nan = final_df['fwd_ret_excess'].isna().any()
print(f"Are there any NaN values in 'fwd_ret_excess'?: {has_nan}")

# Method 4: Filtering rows with NaN values in 'fwd_ret_excess'
nan_rows = final_df[final_df['fwd_ret_excess'].isna()]
print("Rows with NaN values in 'fwd_ret_excess':")
print(nan_rows)


In [ ]:
# Fill missing values
final_df = final_df.fillna(-2)

print(final_df.index)
print(final_df.shape)
print(final_df)


In [ ]:

# Define the columns to select
columns_to_select = characteristics + ["ret_excess", "fwd_ret_excess"]

# Convert specified columns to float16 directly in the original dataframe
for column in columns_to_select:
    final_df[column] = final_df[column].astype('float16')

print(final_df)

**load again**

In [ ]:
# Step 1: Install and configure Kaggle API
!pip install kaggle

import os
os.makedirs('/root/.kaggle/', exist_ok=True)
!cp /content/drive/My\ Drive/Tidy\ Finance\ ML\ dataset/kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets list

# Step 2: Download the dataset from Kaggle
!kaggle datasets download -d hansi666/betadata2 -p /content
!unzip /content/betadata2.zip -d /content/dataset

# Step 3: Load the Parquet file into a DataFrame
import pandas as pd

file_path = '/content/dataset/betadata2.parquet'
final_df = pd.read_parquet(file_path)

# Display the DataFrame to verify
print(final_df)


In [ ]:
# Step 1: Install and configure Kaggle API (if not already done)
!pip install kaggle

import os
os.makedirs('/root/.kaggle/', exist_ok=True)
!cp /content/drive/My\ Drive/Tidy\ Finance\ ML\ dataset/kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets list

# Step 2: Download the dataset from Kaggle
!kaggle datasets download -d hansi666/betadata2 -p /content
!unzip /content/betadata2.zip -d /content/dataset

# Step 3: Upgrade pyarrow and pandas to the latest version
!pip install --upgrade pyarrow pandas

# Step 4: Install fastparquet
!pip install fastparquet

# Step 5: Load the Parquet file into a DataFrame
import pandas as pd

file_path = '/content/dataset/betadata2.parquet'
final_df = pd.read_parquet(file_path, engine='fastparquet')

# Display the DataFrame to verify
print(final_df.head())


In [ ]:
# Display the DataFrame to verify
print(final_df)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# File path to the Parquet file
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/beta3.parquet'


# Now save to Parquet again
final_df.to_parquet(file_path)


In [ ]:
import pandas as pd
from sklearn.preprocessing import quantile_transform

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/CA1000')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True)

# Assuming `df` is your original DataFrame
data = df.copy()


# Assuming 'data' is your DataFrame with a MultiIndex ['date', 'permno']

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]
# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]

# Rename the 'month' column to 'date'
data = data.rename(columns={'month': 'date'})

# Ensure 'data' has a MultiIndex ['date', 'permno']
data = data.set_index(['date', 'permno'])

# Sort the MultiIndex
data = data.sort_index()

# Select the 'ret_excess' column, convert it to a DataFrame, and rename the column to 'returns'
returns = data[['ret_excess']].rename(columns={'ret_excess': 'returns'})

# Filter for dates starting from '1993'
returns = returns.loc[pd.IndexSlice['1993':, :], :]

# Ensure the characteristics DataFrame also has the correct MultiIndex
characteristics_data = data[characteristics]

# Join the returns DataFrame with the characteristics DataFrame
# Since 'returns' is already a DataFrame, no need to stack again
data_combined = returns.join(characteristics_data, how='left')

# Create forward returns
data_combined['returns_fwd'] = data_combined['returns'].unstack('permno').shift(-1).stack(dropna=False)

# Check the number of permno per date
permno_count_per_date = data_combined.groupby(level='date').size()
print(permno_count_per_date.head())

# Normalize the characteristics columns
data_combined.loc[:, characteristics] = (data_combined.loc[:, characteristics]
                                          .groupby(level='date')
                                          .apply(lambda x: pd.DataFrame(quantile_transform(x,
                                                                                           copy=True,
                                                                                           n_quantiles=x.shape[0]),
                                                                        columns=characteristics,
                                                                        index=x.index.get_level_values('permno')))
                                         .mul(2).sub(1))

# Clip the returns columns
data_combined.loc[:, ['returns', 'returns_fwd']] = data_combined.loc[:, ['returns', 'returns_fwd']].clip(lower=-1, upper=1.0)

# Fill missing values
data_combined = data_combined.fillna(-2)

# Save the combined data to HDF5
data_combined.to_hdf(results_path / 'autoencoder.h5', 'model_data')

# Load the combined data for further processing
data_combined = pd.read_hdf(results_path / 'autoencoder.h5', 'model_data')

# Display the DataFrame info
data_combined.info(show_counts=True)

# Display the first few rows to verify
print(data_combined.head())


In [ ]:
import pandas as pd

# Assuming 'returns' DataFrame is already defined with the structure you've described

# Rename the 'month' column to 'date'
df = df.rename(columns={'month': 'date'})

# Ensure 'df' has a MultiIndex ['date', 'permno']
df = df.set_index(['date', 'permno'])

# Sort the MultiIndex
df = df.sort_index()

# Split the DataFrame into characteristics and returns
characteristics_df = df[characteristics]
returns_df = df[['ret_excess']]

# Step 1: Unstack the 'permno' level for returns DataFrame
returns_unstacked = returns_df.unstack(level='permno')

# Step 2: Fill NaN values with -2 in the returns DataFrame
returns_unstacked_filled = returns_unstacked.fillna(-2)

# Step 3: Stack the returns DataFrame back
returns_stacked = returns_unstacked_filled.stack(level='permno')

# Rename the column back to 'ret_excess'
returns_stacked.name = 'ret_excess'

# Convert the stacked returns DataFrame to a DataFrame for joining
returns_stacked_df = returns_stacked.reset_index()

# Convert the original characteristics DataFrame to a DataFrame for joining
characteristics_df = characteristics_df.reset_index()

# Step 4: Perform a left join to merge the characteristics DataFrame back with the modified returns DataFrame
final_df = pd.merge(returns_stacked_df, characteristics_df, on=['date', 'permno'], how='left')

# Set the index back to MultiIndex for consistency
final_df.set_index(['date', 'permno'], inplace=True)

# Display the final DataFrame with NaNs filled in the returns column
print("Final DataFrame with NaNs filled in 'ret_excess':")
print(final_df)

# Count the number of unique permnos for each date in the final DataFrame
permno_counts = final_df.reset_index().groupby('date')['permno'].nunique()

# Check if the number of permnos is the same for each date
is_same_permno_count = permno_counts.nunique() == 1

# Display the result
print("Number of unique permnos for each date:")
print(permno_counts)
print("\nIs the number of permnos the same for each date?:", is_same_permno_count)


In [ ]:
print(final_df.index)

In [ ]:
import pandas as pd
from sklearn.preprocessing import quantile_transform

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/CA1000')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True)

# Assuming `df` is your original DataFrame
data = df.copy()


# Assuming 'data' is your DataFrame with a MultiIndex ['date', 'permno']

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]
# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]

# Rename the 'month' column to 'date'
data = data.rename(columns={'month': 'date'})

# Ensure 'data' has a MultiIndex ['date', 'permno']
data = data.set_index(['date', 'permno'])

# Sort the MultiIndex
data = data.sort_index()

# Select the 'ret_excess' column, convert it to a DataFrame, and rename the column to 'returns'
returns = data[['ret_excess']].rename(columns={'ret_excess': 'returns'})

# Filter for dates starting from '1993'
returns = returns.loc[pd.IndexSlice['1993':, :], :]

# Ensure the characteristics DataFrame also has the correct MultiIndex
characteristics_data = data[characteristics]

# Join the returns DataFrame with the characteristics DataFrame
# Since 'returns' is already a DataFrame, no need to stack again
data_combined = returns.join(characteristics_data, how='left')

# Create forward returns
data_combined['returns_fwd'] = data_combined['returns'].unstack('permno').shift(-1).stack(dropna=False)

# Check the number of permno per date
permno_count_per_date = data_combined.groupby(level='date').size()
print(permno_count_per_date.head())

# Normalize the characteristics columns
data_combined.loc[:, characteristics] = (data_combined.loc[:, characteristics]
                                          .groupby(level='date')
                                          .apply(lambda x: pd.DataFrame(quantile_transform(x,
                                                                                           copy=True,
                                                                                           n_quantiles=x.shape[0]),
                                                                        columns=characteristics,
                                                                        index=x.index.get_level_values('permno')))
                                         .mul(2).sub(1))

# Clip the returns columns
data_combined.loc[:, ['returns', 'returns_fwd']] = data_combined.loc[:, ['returns', 'returns_fwd']].clip(lower=-1, upper=1.0)

# Fill missing values
data_combined = data_combined.fillna(-2)

# Save the combined data to HDF5
data_combined.to_hdf(results_path / 'autoencoder.h5', 'model_data')

# Load the combined data for further processing
data_combined = pd.read_hdf(results_path / 'autoencoder.h5', 'model_data')

# Display the DataFrame info
data_combined.info(show_counts=True)

# Display the first few rows to verify
print(data_combined.head())


In [ ]:
import pandas as pd

# Assuming 'returns' DataFrame is already defined with the structure you've described

# Step 1: Unstack the 'permno' level
returns_unstacked = data.unstack(level='permno')

# Step 2: Fill NaN values with -2
returns_unstacked_filled = returns_unstacked.fillna(-2)

# Step 3: Stack the DataFrame back
returns_stacked = returns_unstacked_filled.stack(level='permno')

# Ensure the column names are preserved
returns_stacked.columns = returns.columns

# Display the stacked DataFrame with NaNs filled
print("Stacked DataFrame with NaNs filled:")
print(returns_stacked)


In [ ]:
print(data)

In [ ]:
# Assuming `df` is your original DataFrame
data = df.copy()

# Rename the 'month' column to 'date'
data = data.rename(columns={'month': 'date'})

# Ensure 'data' has a MultiIndex ['date', 'permno']
data = data.set_index(['date', 'permno'])

# Sort the MultiIndex
data = data.sort_index()

# Select the 'ret_excess' column, convert it to a DataFrame, and rename the column to 'returns'
returns = data[['ret_excess']]

print(returns)
# Stack the returns data
# returns_stacked = returns.stack(dropna=False).to_frame('returns')
# print(returns_stacked)

In [ ]:
print(data.info())

In [ ]:
import pandas as pd

# Assuming 'returns' DataFrame is already defined

# Unstack the 'returns' DataFrame to pivot 'permno' into columns
returns_unstacked = returns.unstack(level='permno')

# Remove the additional 'returns' level from the columns
returns_unstacked.columns = returns_unstacked.columns.droplevel(0)

# Display the unstacked DataFrame
print("Unstacked DataFrame:")
print(returns_unstacked)

# Step 1: Fill NaN values with -2
returns_unstacked_filled = returns_unstacked.fillna(-2)

# Step 2: Stack the DataFrame back
returns_stacked = returns_unstacked_filled.stack()

# Ensure the correct index names are restored after stacking
returns_stacked.index.names = ['date', 'permno']

# Step 3: Rename the resulting Series
returns_stacked.name = "ret_excess"

# Display the stacked DataFrame with the new column name
print("\nStacked DataFrame with NaNs filled and renamed column:")
print(returns_stacked)


In [ ]:
import pandas as pd

# Unstack the 'returns' DataFrame to pivot 'permno' into columns
returns_unstacked = returns.unstack(level='permno')

# Remove the additional 'returns' level from the columns
returns_unstacked.columns = returns_unstacked.columns.droplevel(0)

# Display the unstacked DataFrame
print(returns_unstacked)

# Assuming 'returns' DataFrame is already defined and unstacked as 'returns_unstacked'
# Step 1: Fill NaN values with -2
returns_unstacked_filled = returns_unstacked.fillna(-2)

# Step 2: Stack the DataFrame back
returns_stacked = returns_unstacked_filled.stack()

# Display the stacked DataFrame
print(returns_stacked)


In [ ]:
print(returns_stacked)

In [ ]:
import matplotlib.pyplot as plt

# Count the number of non-NaN stocks for each date
non_nan_counts = returns_unstacked.notna().sum(axis=1)

# Plot the result
plt.figure(figsize=(14, 7))
plt.plot(non_nan_counts.index, non_nan_counts.values, label='Number of Non-NaN Stocks')
plt.xlabel('Date')
plt.ylabel('Number of Non-NaN Stocks')
plt.title('Number of Non-NaN Stocks Over Time')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Convert index to datetime if it's not already
returns_unstacked.index = pd.to_datetime(returns_unstacked.index)

# Calculate the total number of days in the entire time period
total_days = (returns_unstacked.index.max() - returns_unstacked.index.min()).days + 1

# Calculate the number of non-NaN values for each permno over the entire time period
non_nan_counts_total = returns_unstacked.count()

# Calculate the percentage of observations for each permno over the entire time period
percent_observations_total = (non_nan_counts_total / total_days) * 100

# Filter out permnos with less than 15 percent of observations over the entire time period
filtered_permnos = percent_observations_total[percent_observations_total >= 15].index

# Filter the original returns_unstacked dataframe to keep only the filtered permnos
returns_filtered = returns_unstacked[filtered_permnos]

# Count the number of non-NaN stocks for each date in the filtered dataset
non_nan_counts_filtered = returns_filtered.notna().sum(axis=1)

# Plot the result
plt.figure(figsize=(14, 7))
plt.plot(non_nan_counts_filtered.index, non_nan_counts_filtered.values, label='Number of Non-NaN Stocks (Filtered)')
plt.xlabel('Date')
plt.ylabel('Number of Non-NaN Stocks')
plt.title('Number of Non-NaN Stocks Over Time (Filtered for Entire Period)')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
print(returns)

In [ ]:
# Reset the index to make 'date' and 'permno' columns
returns_reset = returns.reset_index(level=['date', 'permno'])

# Group by 'date' and count unique 'permno' values
unique_permno_counts = returns_reset.groupby('date')['permno'].nunique()

# Convert to a DataFrame
unique_permno_counts_df = unique_permno_counts.to_frame(name='unique_permno_count').reset_index()

# Display the result
print(unique_permno_counts_df)


In [ ]:
import pandas as pd
from sklearn.preprocessing import quantile_transform
from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/CA1000')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True)

# Load the returns data
returns = pd.read_hdf(results_path / 'autoencoder.h5', 'returns')

# Stack the returns data
returns = returns.stack(dropna=False).to_frame('returns')

# Filter for dates starting from '1993'
returns = returns.loc[pd.IndexSlice['1993':, :], :]

# Create a MultiIndex with all combinations of dates and tickers
all_dates = returns.index.get_level_values('date').unique()
all_tickers = returns.index.get_level_values('permno').unique()
full_index = pd.MultiIndex.from_product([all_dates, all_tickers], names=['date', 'permno'])

# Reindex the returns DataFrame to ensure all dates and tickers are present
returns = returns.reindex(full_index).fillna(-2)

# Assuming `df` is your original DataFrame
data = df.copy()

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]

# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]

# Rename the 'month' column to 'date'
data = data.rename(columns={'month': 'date'})

# Ensure 'data' has a MultiIndex ['date', 'permno']
data = data.set_index(['date', 'permno'])

# Sort the MultiIndex
data = data.sort_index()

# Ensure the characteristics DataFrame also has the correct MultiIndex
characteristics_data = data[characteristics]

# Join the returns DataFrame with the characteristics DataFrame
data_combined = returns.join(characteristics_data, how='left')

# Fill missing values in the combined data
data_combined = data_combined.fillna(-2)

# Create forward returns
data_combined['returns_fwd'] = data_combined['returns'].unstack('permno').shift(-1).stack(dropna=False)

# Check the number of permno per date
permno_count_per_date = data_combined.groupby(level='date').size()
print(permno_count_per_date.head())

# Normalize the characteristics columns
data_combined.loc[:, characteristics] = (data_combined.loc[:, characteristics]
                                          .groupby(level='date')
                                          .apply(lambda x: pd.DataFrame(quantile_transform(x,
                                                                                           copy=True,
                                                                                           n_quantiles=x.shape[0]),
                                                                        columns=characteristics,
                                                                        index=x.index.get_level_values('permno')))
                                         .mul(2).sub(1))

# Clip the returns columns
data_combined.loc[:, ['returns', 'returns_fwd']] = data_combined.loc[:, ['returns', 'returns_fwd']].clip(lower=-1, upper=1.0)

# Save the combined data to HDF5
data_combined.to_hdf(results_path / 'autoencoder.h5', 'model_data')

# Load the combined data for further processing
data_combined = pd.read_hdf(results_path / 'autoencoder.h5', 'model_data')

# Display the DataFrame info
data_combined.info(show_counts=True)

# Display the first few rows to verify
print(data_combined.head())

# Initialize cross-validation
YEAR = 12
cv = MultipleTimeSeriesCV(n_splits=5,
                          train_period_length=20*YEAR,
                          test_period_length=1*YEAR,
                          lookahead=1)

def get_train_valid_data(data, train_idx, val_idx):
    train, val = data.iloc[train_idx], data.iloc[val_idx]

    # List unique dates for train and val sets
    train_dates = train.index.get_level_values('date').unique()
    val_dates = val.index.get_level_values('date').unique()

    # Create dictionaries to hold the data
    X1_train_dict, X1_val_dict = {}, {}
    X2_train_dict, X2_val_dict = {}, {}
    y_train_dict, y_val_dict = {}, {}

    for date in train_dates:
        daily_data = train.xs(date, level='date')
        n_tickers = len(daily_data)
        X1_train_dict[date] = daily_data[characteristics].values.reshape(1, n_tickers, n_characteristics)
        X2_train_dict[date] = daily_data['returns'].values.reshape(1, n_tickers)
        y_train_dict[date] = daily_data['returns_fwd'].values.reshape(1, n_tickers)

    for date in val_dates:
        daily_data = val.xs(date, level='date')
        n_tickers = len(daily_data)
        X1_val_dict[date] = daily_data[characteristics].values.reshape(1, n_tickers, n_characteristics)
        X2_val_dict[date] = daily_data['returns'].values.reshape(1, n_tickers)
        y_val_dict[date] = daily_data['returns_fwd'].values.reshape(1, n_tickers)

    # Convert dictionaries to lists
    X1_train = list(X1_train_dict.values())
    X2_train = list(X2_train_dict.values())
    y_train = list(y_train_dict.values())
    X1_val = list(X1_val_dict.values())
    X2_val = list(X2_val_dict.values())
    y_val = list(y_val_dict.values())

    return X1_train, X2_train, y_train, X1_val, X2_val, y_val

# Example loop for using cross-validation
for fold, (train_idx, val_idx) in enumerate(cv.split(data_combined)):
    X1_train, X2_train, y_train, X1_val, X2_val, y_val = get_train_valid_data(data_combined, train_idx, val_idx)
    # Example model fitting (model definition not included)
    # model.fit([X1_train, X2_train], y_train)
    # Add your model training code here


In [ ]:
print(returns)

In [ ]:
import pandas as pd
from sklearn.preprocessing import quantile_transform

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/CA1000')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True)



data = pd.read_hdf(results_path / 'autoencoder.h5', 'model_data')

In [ ]:
# Check the number of permno per date
permno_count_per_date = data.groupby(level='date').size()
print(permno_count_per_date.mean())

In [ ]:
# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]
# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]


n_factors = 3
n_characteristics = len(characteristics)
n_tickers = len(data.index.unique('permno'))
print(n_characteristics)
print(n_tickers)

In [ ]:
def make_model(hidden_units=8, n_factors=3):
    input_beta = Input((n_tickers, n_characteristics), name='input_beta')
    input_factor = Input((n_tickers,), name='input_factor')

    hidden_layer = Dense(units=hidden_units, activation='relu', name='hidden_layer')(input_beta)
    batch_norm = BatchNormalization(name='batch_norm')(hidden_layer)

    output_beta = Dense(units=n_factors, name='output_beta')(batch_norm)

    output_factor = Dense(units=n_factors, name='output_factor')(input_factor)

    output = Dot(axes=(2,1), name='output_layer')([output_beta, output_factor])

    model = Model(inputs=[input_beta, input_factor], outputs=output)
    model.compile(loss='mse', optimizer='adam')
    return model



In [ ]:
# YEAR = 12
# cv = MultipleTimeSeriesCV(n_splits=5, train_period_length=20*YEAR, val_period_length=2*YEAR, test_period_length=1*YEAR, lookahead=0)

In [ ]:
YEAR = 12
cv = MultipleTimeSeriesCV(n_splits=5,
                          train_period_length=20*YEAR,
                          test_period_length=1*YEAR,
                          lookahead=1)

In [ ]:
def get_train_valid_data(data, train_idx, val_idx):
    train, val = data.iloc[train_idx], data.iloc[val_idx]
    X1_train = train.loc[:, characteristics].values.reshape(-1, n_tickers, n_characteristics)
    X1_val = val.loc[:, characteristics].values.reshape(-1, n_tickers, n_characteristics)
    X2_train = train.loc[:, 'returns'].unstack('permno')
    X2_val = val.loc[:, 'returns'].unstack('permno')
    y_train = train.returns_fwd.unstack('permno')
    y_val = val.returns_fwd.unstack('permno')
    return X1_train, X2_train, y_train, X1_val, X2_val, y_val

In [ ]:
def get_train_valid_test_data(merged_data, train_idx_merged, val_idx_merged, test_idx_merged):
    # Process merged_df_padded data
    train_merged, val_merged, test_merged = merged_data.iloc[train_idx_merged], merged_data.iloc[val_idx_merged], merged_data.iloc[test_idx_merged]

    X1_train = train_merged.loc[:, characteristics].values.reshape(-1, n_tickers, len(n_characteristics))
    X1_val = val_merged.loc[:, characteristics].values.reshape(-1, n_tickers, len(n_characteristics))
    X1_test = test_merged.loc[:, characteristics].values.reshape(-1, n_tickers, len(n_characteristics))


    X2_train = train_merged.ret_excess.unstack('permno')
    X2_val = val_merged.returns.unstack('permno')
    X2_test = test_merged.returns.unstack('permno')

    y_train = train_merged.returns.unstack('permno')
    y_val = val_merged.returns.unstack('permno')
    y_test = test_merged.returns.unstack('permno')

#    Process loaded_portfolio_rets data
#    print("Portfolio data index:", portfolio_data.index)

    return X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test


In [ ]:
from time import time
factor_opts = [2, 3, 4, 5, 6]
unit_opts = [8, 16, 32]
param_grid = list(product(unit_opts, factor_opts))
#Run Cross-Validation
batch_size = 32
cols = ['units', 'n_factors', 'fold', 'epoch', 'ic_mean',
        'ic_daily_mean', 'ic_daily_std', 'ic_daily_median']
start = time()
for units, n_factors in param_grid:
    scores = []
    model = make_model(hidden_units=units, n_factors=n_factors)
    for fold, (train_idx, val_idx) in enumerate(cv.split(data)):
        X1_train, X2_train, y_train, X1_val, X2_val, y_val= get_train_valid_data(data, train_idx, val_idx)
        for epoch in range(250):
            model.fit([X1_train, X2_train], y_train,
                      batch_size=batch_size,
                      validation_data=([X1_val, X2_val], y_val),
                      epochs=epoch + 1,
                      initial_epoch=epoch,
                      verbose=0, shuffle=True)
            result = (pd.DataFrame({'y_pred': model.predict([X1_val,
                                                             X2_val]).reshape(-1),
                                    'y_true': y_val.stack().values},
                                   index=y_val.stack().index)
                      .replace(-2, np.nan).dropna())
            r0 = spearmanr(result.y_true, result.y_pred)[0]
            r1 = result.groupby(level='date').apply(lambda x: spearmanr(x.y_pred,
                                                                        x.y_true)[0])

            scores.append([units, n_factors, fold, epoch, r0,
                           r1.mean(), r1.std(), r1.median()])
            if epoch % 50 == 0:
                print(f'{format_time(time()-start)} | {n_factors} | {units:02} | {fold:02}-{epoch:03} | {r0:6.2%} | '
                  f'{r1.mean():6.2%} | {r1.median():6.2%}')
    scores = pd.DataFrame(scores, columns=cols)
    scores.to_hdf(results_path / 'scores.h5', f'{units}/{n_factors}')

In [ ]:
# def get_train_valid_data_combined(merged_data, portfolio_data, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio):
#     # Process merged_df_padded data
#     train_merged, val_merged, test_merged = merged_data.iloc[train_idx_merged], merged_data.iloc[val_idx_merged], merged_data.iloc[test_idx_merged]

#     X1_train = train_merged.loc[:, characteristics].values.reshape(-1, n_tickers, n_characteristics)
#     X1_val = val_merged.loc[:, characteristics].values.reshape(-1, n_tickers, n_characteristics)
#     X1_test = test_merged.loc[:, characteristics].values.reshape(-1, n_tickers, n_characteristics)
#     # print(train_merged.index.get_level_values('date').unique())
#     # print(val_merged.index.get_level_values('date').unique())
#     # print(test_merged.index.get_level_values('date').unique())

#     y_train = train_merged.ret_excess.unstack('permno')
#     y_val = val_merged.ret_excess.unstack('permno')
#     y_test = test_merged.ret_excess.unstack('permno')

#     # Process loaded_portfolio_rets data
#     X2_train = portfolio_data.loc[train_dates_portfolio]
#     X2_val = portfolio_data.loc[val_dates_portfolio]
#     X2_test = portfolio_data.loc[test_dates_portfolio]

#     # Process loaded_portfolio_rets data
#     # print("Train dates portfolio:", train_dates_portfolio)
#     # print("Validation dates portfolio:", val_dates_portfolio)
#     # print("Test dates portfolio:", test_dates_portfolio)

# #    print("Portfolio data index:", portfolio_data.index)

#     return X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test
